In [ ]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words = vocab_size)

In [ ]:
print('리뷰의 최대 길이 : {}'.format(max(len(l) for l in X_train)))
print('리뷰의 평균 길이 : {}'.format(sum(map(len, X_train))/len(X_train)))

In [ ]:
max_len = 500
X_train = pad_sequences(X_train, maxlen=max_len, padding='post')
X_test = pad_sequences(X_test, maxlen=max_len, padding='post')

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Concatenate, Dropout
from tensorflow.keras import Input, Model
from tensorflow.keras import optimizers
import os
import time

In [ ]:
class BahdanauAttention(tf.keras.Model):
  def __init__(self, units):
    super(BahdanauAttention, self).__init__()
    self.W1 = Dense(units)
    self.W2 = Dense(units)
    self.V = Dense(1)

  def call(self, values, query): # 단, key와 value는 같음
    # query shape == (batch_size, hidden size)
    hidden_with_time_axis = tf.expand_dims(query, 1)

    # score shape == (batch_size, max_length, 1)
    score = self.V(tf.nn.tanh(
        self.W1(values) + self.W2(hidden_with_time_axis)))

    # attention_weights shape == (batch_size, max_length, 1)
    attention_weights = tf.nn.softmax(score, axis=1)

    # context_vector shape after sum == (batch_size, hidden_size)
    context_vector = attention_weights * values
    context_vector = tf.reduce_sum(context_vector, axis=1)

    return context_vector, attention_weights

In [ ]:
sequence_input = Input(shape=(max_len,), dtype='int32')
embedded_sequences = Embedding(vocab_size, 128, input_length=max_len, mask_zero = True)(sequence_input)

lstm = Bidirectional(LSTM(64, dropout=0.5, return_sequences = True, use_cudnn=False))(embedded_sequences)

lstm, forward_h, forward_c, backward_h, backward_c = Bidirectional \
  (LSTM(64, dropout=0.5, return_sequences=True, return_state=True, use_cudnn=False))(lstm)

print(lstm.shape, forward_h.shape, forward_c.shape, backward_h.shape, backward_c.shape)

state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])

attention = BahdanauAttention(64)
context_vector, attention_weights = attention(lstm, state_h)

dense1 = Dense(20, activation="relu")(context_vector)
dropout = Dropout(0.5)(dense1)
output = Dense(1, activation="sigmoid")(dropout)
model = Model(inputs=sequence_input, outputs=output)

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
EPOCHS = 3
BATCH_SIZE = 256

In [ ]:
print("\n===== Bahdanau Attention 모델 학습 시작 =====")
start_time_bah = time.time()
history_bah = model.fit(
    X_train, y_train,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1
)
time_bah = time.time() - start_time_bah

test_loss_bah, test_acc_bah = model.evaluate(X_test, y_test, verbose=0)
train_acc_bah = history_bah.history['accuracy'][-1]

print("\n[ Bahdanau Attention 결과 ]")
print(" - 마지막 epoch train accuracy : {:.4f}".format(train_acc_bah))
print(" - test accuracy               : {:.4f}".format(test_acc_bah))
print(" - 총 학습 시간                : {:.2f}초".format(time_bah))

In [ ]:
class DotProductAttention(tf.keras.Model):
  def __init__(self):
    super(DotProductAttention, self).__init__()

  def call(self, values, query):
    """
    values: (batch_size, time_steps, hidden_dim)
    query : (batch_size, hidden_dim)
    """
    # (batch_size, hidden_dim, 1)
    query_with_time_axis = tf.expand_dims(query, 2)

    # score: (batch_size, time_steps, 1)
    score = tf.matmul(values, query_with_time_axis)

    # Scaled Dot-Product: hidden_dim으로 나눠서 스케일링
    d_k = tf.cast(tf.shape(values)[-1], tf.float32)
    score = score / tf.math.sqrt(d_k)

    # attention_weights: (batch_size, time_steps, 1)
    attention_weights = tf.nn.softmax(score, axis=1)

    # context_vector: (batch_size, hidden_dim)
    context_vector = attention_weights * values
    context_vector = tf.reduce_sum(context_vector, axis=1)

    return context_vector, attention_weights

In [ ]:
sequence_input2 = Input(shape=(max_len,), dtype='int32')
embedded_sequences2 = Embedding(vocab_size, 128, input_length=max_len, mask_zero = True)(sequence_input2)

lstm2 = Bidirectional(LSTM(64, dropout=0.5, return_sequences = True, use_cudnn=False))(embedded_sequences2)

lstm2, forward_h2, forward_c2, backward_h2, backward_c2 = Bidirectional \
  (LSTM(64, dropout=0.5, return_sequences=True, return_state=True, use_cudnn=False))(lstm2)

state_h2 = Concatenate()([forward_h2, backward_h2]) # 은닉 상태
state_c2 = Concatenate()([forward_c2, backward_c2]) # 셀 상태

attention2 = DotProductAttention()
context_vector2, attention_weights2 = attention2(lstm2, state_h2)

dense1_2 = Dense(20, activation="relu")(context_vector2)
dropout2 = Dropout(0.5)(dense1_2)
output2 = Dense(1, activation="sigmoid")(dropout2)
model2 = Model(inputs=sequence_input2, outputs=output2)

model2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
print("\n===== Dot-Product Attention 모델 학습 시작 =====")
start_time_dot = time.time()
history_dot = model2.fit(
    X_train, y_train,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1
)
time_dot = time.time() - start_time_dot

test_loss_dot, test_acc_dot = model2.evaluate(X_test, y_test, verbose=0)
train_acc_dot = history_dot.history['accuracy'][-1]

print("\n[ Dot-Product Attention 결과 ]")
print(" - 마지막 epoch train accuracy : {:.4f}".format(train_acc_dot))
print(" - test accuracy               : {:.4f}".format(test_acc_dot))
print(" - 총 학습 시간                : {:.2f}초".format(time_dot))


In [ ]:
print("\n========== 최종 비교 ==========")
print("Bahdanau  : train_acc = {:.4f}, test_acc = {:.4f}, time = {:.2f}초".format(
    train_acc_bah, test_acc_bah, time_bah
))
print("Dot-Product: train_acc = {:.4f}, test_acc = {:.4f}, time = {:.2f}초".format(
    train_acc_dot, test_acc_dot, time_dot
))